# Install Dependencies

In [ ]:
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.1/117.1 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 375.8/375.8 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 MB 36.9 MB/s eta 0:00:00


In [ ]:
!pip install -q datasets
!pip install -q regex

In [ ]:
!pip install -q emoji
!pip install -q PyArabic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 8.8 MB/s eta 0:00:00


In [ ]:
!pip install -q diffusers

# login

In [ ]:
import huggingface_hub
huggingface_hub.login('HF_TOKEN')

# Import Required Modules

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ['CUDA_LAUNCH_BLOCKING']="1"
os.environ['TORCH_USE_CUDA_DSA'] = "1"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import pandas as pd
import random
from sklearn.utils import shuffle
import os
import re
from tqdm import tqdm
import bitsandbytes as bnb
import torch
import torch.nn as nn
import transformers
from datasets import Dataset
from peft import LoraConfig, PeftConfig
from trl import SFTTrainer
from transformers import (AutoModelForCausalLM,
                          AutoTokenizer,
                          BitsAndBytesConfig,
                          TrainingArguments,
                          pipeline,
                          logging)
from sklearn.metrics import (accuracy_score,
                             classification_report,
                             precision_score,
                             recall_score,
                             f1_score,
                             confusion_matrix)
from sklearn.model_selection import train_test_split
import emoji
import pyarabic.araby as araby

In [ ]:
import pandas as pd

In [ ]:
import torch
import torch.distributed as dist

# Load Model

In [ ]:
model_name = "Qwen/Qwen2.5-7B-Instruct"

compute_dtype = getattr(torch, "float16")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    # quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=True,
)

model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(model_name,
                                          trust_remote_code=True,
                                          padding_side="left",
                                          add_eos_token=True,
                                         )

# Assign pad_token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "left"

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

# Load Data

In [ ]:
import pandas as pd
data = pd.read_excel('sampled_sentiment_data.xlsx')

# Zero Shot

## Predict Using Arabic Prompt

In [ ]:
prompt = '''الجملة:
لقد ضلم ماكرون الشعب المصرى وساهم في قمعه وقتله بتنسيقه مع العصابة السيسية ودعمها فانتقم الله منه https://t.co/anjidHMCzK
 المشاعر المتوقعة:'''
messages = [
    {"role": "system", "content": "أنت مساعد ذكاء اصطناعي متخصص في تحليل المشاعر. صنّف مشاعر الجملة التالية بناءً على نبرتها العاطفية. اختر شعورًا واحدًا فقط من بين: إيجابي، سلبي، أو محايد."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=512
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

In [ ]:
zero_pred = []
for text in data['Text']:
    prompt = f'''الجملة:
    {text}
    المشاعر المتوقعة:'''
    messages = [
        {"role": "system", "content": "أنت مساعد ذكاء اصطناعي متخصص في تحليل المشاعر. صنّف مشاعر الجملة التالية بناءً على نبرتها العاطفية. اختر شعورًا واحدًا فقط من بين: إيجابي، سلبي، أو محايد."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    zero_pred.append(response)

In [ ]:
pred_zero = pd.DataFrame()
pred_zero['Predicted'] = zero_pred
pred_zero['Predicted'].value_counts()

,count
Predicted,
إيجابي,197
محايد,31
سلبي,23
المشاعر: سلبي,23
المشاعر: إيجابي,10
...,...
المشاعر العاطفية في هذه الجملة تبدو محايدة. الجملة من الشعر العربي وتبدو أكثر تعقيداً في المعنى والمشاعر التي تعبّر عنها، لكن بشكل عام لا تظهر أي مشاعر سلبية أو إيجابية واضحة.,1
إيجابي\n\nالجملة تحمل نبرة إيجابية لأنها تشير إلى تنافس قوي بين لاعبين، مما يدل على مستوى عالٍ من اللعب والحماس الرياضي.,1
المشاعر: سلبي\n\nالجملة تحتوي على قلق بشأن مشكلة صحية بسيطة (闯入眼睛的消毒液)，这表达了某种程度上的不安或担忧，因此可以被分类为消极情绪。,1


In [ ]:
nor_pre = []
for pr in pred_zero['Predicted']:
  if "سلبي" in pr:
    nor_pre.append("Negative")
  elif "سلبية" in pr:
    nor_pre.append("Negative")
  elif "Negative" in pr:
    nor_pre.append("Negative")
  elif "إيجابي" in pr:
    nor_pre.append("Positive")
  elif "إيجابية" in pr:
    nor_pre.append("Positive")
  elif "ايجابي" in pr:
    nor_pre.append("Positive")
  elif "ايجابية" in pr:
    nor_pre.append("Positive")
  elif "محايد" in pr:
    nor_pre.append("Neutral")
  elif "محايدة" in pr:
    nor_pre.append("Neutral")
  elif "محيادي" in pr:
    nor_pre.append("Neutral")
  elif "محاييد" in pr:
    nor_pre.append("Neutral")
  elif "محيود" in pr:
    nor_pre.append("Neutral")
  elif "المحيد" in pr:
    nor_pre.append("Neutral")
  else:
    nor_pre.append("Unclassified")

pred_zero['Normalized Sentiment'] = nor_pre

In [ ]:
pred_zero['Normalized Sentiment'].value_counts()

,count
Normalized Sentiment,
Positive,244
Negative,175
Neutral,81
Unclassified,1


In [ ]:
pred_zero['Text'] = data['Text']
pred_zero.to_excel('Qwen2.5-SA-ZeroShot.xlsx', index = False)

In [ ]:
y_true = data['sentiment'].values
print(classification_report(y_true, pred_zero['Normalized Sentiment'].values, digits = 4))

              precision    recall  f1-score   support

    Negative     0.6686    0.7006    0.6842       167
     Neutral     0.6049    0.2934    0.3952       167
    Positive     0.6270    0.9162    0.7445       167
Unclassified     0.0000    0.0000    0.0000         0

    accuracy                         0.6367       501
   macro avg     0.4751    0.4775    0.4560       501
weighted avg     0.6335    0.6367    0.6080       501




## Predict Using English Prompt

In [ ]:
zero_pred = []
for text in data['Text']:
    prompt = f'''Sentence:
    {text}
    Predicted Sentiment'''
    messages = [
        {"role": "system", "content": "You are an AI assistant specialized in sentiment analysis. Classify the sentiment of the following sentence based on its emotional tone. Choose only one sentiment between: Positive, Negative, or Neutral."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    zero_pred.append(response)

In [ ]:
pred_zero = pd.DataFrame()
pred_zero['Predicted'] = zero_pred
pred_zero['Predicted'].value_counts()

,count
Predicted,
Positive,179
Negative,159
Neutral,131
"Neutral\n\nThis sentence appears to be making a request or demand for fairness and action from a specific authority (the Minister of Awqaf), without expressing strong positive or negative emotions. It is more of an informational statement seeking a change in policy or practice.",1
"Neutral\n\nThe sentence is a prayer expressing wishes for someone who has passed away and includes blessings. While it contains elements that could be seen as positive (praying for someone to have paradise), the overall tone is not strongly emotional or expressing clear happiness or sadness. The inclusion of a sad face emoji at the end might suggest some underlying emotion but does not dominate the sentiment of the prayer itself.",1
Neutral\n\nThe sentence appears to be in Arabic and seems to express a request or plea for assistance in finding employment in Jordan instead of staying in Egypt due to poverty. It does not strongly convey positive or negative emotions but rather a statement of circumstance or hope.,1
"Neutral\n\nThe sentence translates to ""It's normal to break up and get back with him; it suits him anyway, you've been with someone else before getting married."" While there is a hint of casualness or acceptance in the statement, it doesn't clearly convey strong positive or negative emotions, hence the classification as neutral.",1
"Neutral\n\nThe sentence is a neutral statement about someone being ahead of everyone else regarding exclusive news, without any clear positive or negative emotional tone.",1
"Neutral\n\nThis sentence has a light-hearted and somewhat humorous tone, but it doesn't express strong positive or negative emotions. The use of ""😂"" at the end indicates a playful or mildly amused tone rather than a clearly positive or negative sentiment.",1


In [ ]:
nor_pre = []
for pr in pred_zero['Predicted']:
  if "Negative" in pr:
    nor_pre.append("Negative")
  elif "Positive" in pr:
    nor_pre.append("Positive")
  elif "Neutral" in pr:
    nor_pre.append("Neutral")
  else:
    nor_pre.append("Unclassified")

pred_zero['Normalized Sentiment'] = nor_pre

In [ ]:
pred_zero['Text'] = data['Text']
pred_zero.to_excel('Qwen2.5-SA-ZeroShot-English-Prompt.xlsx', index = False)

In [ ]:
y_true = data['sentiment'].values
print(classification_report(y_true, pred_zero['Normalized Sentiment'].values, digits = 4))

              precision    recall  f1-score   support

    Negative     0.7625    0.7305    0.7462       167
     Neutral     0.6173    0.5988    0.6079       167
    Positive     0.7207    0.7725    0.7457       167

    accuracy                         0.7006       501
   macro avg     0.7002    0.7006    0.6999       501
weighted avg     0.7002    0.7006    0.6999       501



# Pred Few Shot

In [ ]:
content = '''
You are an AI assistant specialized in sentiment analysis. Classify the sentiment of the following sentence based on its emotional tone. Choose only one sentiment between: Positive, Negative, or Neutral.

Example 1:
كل فرحة تصنعها لغيرك ستعود لك بشكل اجمل صباح الخير ..

Sentiment: Positive

Example 2:
سئمت رؤيتكِ في كل أغنية أسمعها في كل شعر أقرأه في منتصف قهوتي في كل خطوة أخطيها في كل دمعة سئمت

Sentiment: Negative

Example 3:
اذا لم تستطع أن تترك اثرا جميلا في القلوب فلا تزرع فيها آلما لا ينسى.

Sentiment: Neutral
'''

few_pred = []
for text in data['Text']:
    prompt = f'''The sentence you need to classify
    Sentence:
    {text}
    Predicted Sentiment'''
    messages = [
        {"role": "system", "content": content},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    few_pred.append(response)

In [ ]:
pred_few = pd.DataFrame()
pred_few['Predicted'] = few_pred
pred_few['Predicted'].value_counts()

,count
Predicted,
Negative,165
Positive,93
Neutral,17
"Neutral\n\nThis sentence expresses a situation where the speaker is in a difficult position and needs help, which leans more towards a neutral tone rather than positive or negative. It's a statement of current circumstances without strong emotional indicators.",1
"Neutral\n\nThis sentence appears to be discussing health issues and stress related to building a house, which could be seen as a slightly negative topic. However, the overall tone does not express strong emotions like sadness or anger, and it seems more informational or observational rather than explicitly positive or negative. Therefore, it is best classified as neutral.",1
...,...
Neutral\n\nThis sentence presents a straightforward comparison of goals scored by different players and teams without expressing strong positive or negative emotions.,1
"Positive\n\nThe sentence appears to describe a joyful and lively moment with expressions of happiness and excitement, indicated by words and emojis that convey positive emotions.",1
"Neutral\n\nThis sentence appears to be a query or concern about an issue (getting sanitizer in their eyes) and seems to be seeking advice, which generally carries a neutral tone without strong positive or negative emotions.",1


In [ ]:
nor_pre = []
for pr in pred_few['Predicted']:
  if "Negative" in pr:
    nor_pre.append("Negative")
  elif "Positive" in pr:
    nor_pre.append("Positive")
  elif "Neutral" in pr:
    nor_pre.append("Neutral")
  else:
    nor_pre.append("Unclassified")

pred_few['Normalized Sentiment'] = nor_pre

In [ ]:
pred_few['Normalized Sentiment'].value_counts()

,count
Normalized Sentiment,
Negative,170
Positive,169
Neutral,162


In [ ]:
pred_few['Text'] = data['Text']
pred_few.to_excel('Qwen2.5-SA-FewShot-English-Prompt.xlsx', index = False)

In [ ]:
print(classification_report(y_true, pred_few['Normalized Sentiment'].values, digits = 4))

              precision    recall  f1-score   support

    Negative     0.7636    0.7545    0.7590       167
     Neutral     0.5976    0.6048    0.6012       167
    Positive     0.7485    0.7485    0.7485       167

    accuracy                         0.7026       501
   macro avg     0.7033    0.7026    0.7029       501
weighted avg     0.7033    0.7026    0.7029       501



# CoT

In [ ]:
content = '''
You are an AI assistant specialized in sentiment analysis. Classify the sentiment of the following sentence based on its emotional tone. Choose only one sentiment between: Positive, Negative, or Neutral.

Step 1: Read the sentence
Carefully read the sentence to fully understand its meaning, context, and tone. Consider both explicit statements and any implied emotional cues.

Step 2: Identify Emotionally Charged Language
- Highlight positive language, such as words indicating satisfaction, happiness, or praise (e.g., great, amazing, love, well-done).
- Highlight negative language, such as words indicating dissatisfaction, frustration, or criticism (e.g., terrible, hate, broken, disappointing).
- Neutral statements neither praise nor criticize.

Step 3: Analyze the Emotional Balance
Consider the overall tone and intent of the sentence, including sarcasm or contrast.

Step 4: Determine Sentiment
- If positive sentiment dominates, classify as Positive.
- If negative sentiment dominates, classify as Negative.
- If there is no clear emotional direction or the content is purely factual, classify as Neutral.

Example 1:
كل فرحة تصنعها لغيرك ستعود لك بشكل اجمل صباح الخير ..

Thoughts:
- Positive words: فرحة, اجمل, الخير
- Negative words: None
- Emotional Balance: tweet uses only positive language and promotes an uplifting message about generosity and the return of happiness.
- Sentiment: positive

Predicted Sentiment: Positive

Example 2:
سئمت رؤيتكِ في كل أغنية أسمعها في كل شعر أقرأه في منتصف قهوتي في كل خطوة أخطيها في كل دمعة سئمت


Thoughts:
- Positive word: None
- Negative words: سئمت, دمعة
- Emotional Balance: tweet uses only negative language which revolves around fatigue, sadness, and being overwhelmed by memories.
- Sentiment: Negative

Predicted Sentiment: Negative

Example 3:
اذا لم تستطع أن تترك اثرا جميلا في القلوب فلا تزرع فيها آلما لا ينسى.

Thoughts:
- Positive words: جميلاً
- Negative words: ألماً
- Emotional Balance: tweet mentions both positive and negative outcomes, the overall tone is cautionary and moralistic, not emotionally expressive.
- Sentiment: Neutral

Predicted Sentiment: Neutral
'''

cot_pred = []
for text in data['Text']:
    prompt = f'''The sentence you need to classify
    Sentence:
    {text}
    Predicted Sentiment'''
    messages = [
        {"role": "system", "content": content},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    cot_pred.append(response)

In [ ]:
pred_cot = pd.DataFrame()
pred_cot['Predicted'] = cot_pred
pred_cot['Predicted'].value_counts()

,count
Predicted,
Negative,26
Positive,2
"Negative\n\nThe sentence expresses frustration and disappointment regarding the closure of a disco that came from Dubai, mentioning that the entertainment authority did not approve it or provide any information. The use of ""مادرت عنه"" (disapproved of it) and the exclamation ""ولا باقي!!! "" (and that's it!!!) indicate a negative sentiment. The hashtag ""#موسم_جده"" (Season_Jeddah) suggests this is related to an event or season in Jeddah, adding context to the negative reaction.",1
"Negative\n\nThoughts:\n- Positive words: فرحانه (happy)\n- Negative words: زعلت (annoyed), بيشغلني (occupies me, implying stress or irritation)\n- Emotional Balance: The sentence starts with a positive emotion but shifts to a negative one, expressing annoyance despite being initially happy. The overall tone suggests frustration or irritation.\n\nPredicted Sentiment: Negative",1
Positive\n\nThoughts:\n- Positive words: أحلى (sweetest)\n- Negative words: None\n- Emotional Balance: The sentence expresses a positive sentiment about the fragrance resulting from mixing incense with oud.\n\nPredicted Sentiment: Positive,1
...,...
"Positive\n\nThe sentence highlights the diversity of events in #JeddahSeason and emphasizes that these events are impressive and suitable for all ages. The use of words like ""تنوع"" (diversity) and ""الننبهر"" (amazed) indicate a positive sentiment.",1
"Positive\n\nThe sentence contains several positive elements such as ""تعجبني"" (I like), ""بسيطه"" (simple), ""اخلاقهم"" (their morals), ""الابتسامة"" (smile), and ""التواضع"" (modesty). The use of heart emojis (💙) and a sparkle emoji (✨) further reinforces the positive sentiment. Therefore, the overall sentiment of the sentence is positive.",1
"Predicted Sentiment: Positive\n\nJustification:\n- The sentence includes a smiley face emoticon (🙂🙂), which generally indicates a positive or friendly tone.\n- The overall context suggests a willingness to give others a chance, which can be seen as a positive gesture.\n- While there is a mention of people leaving, the focus is on providing an opportunity for others, which leans more towards a positive outlook.",1


In [ ]:
nor_pre = []
for pr in pred_cot['Predicted']:
  if "Negative" in pr:
    nor_pre.append("Negative")
  elif "Positive" in pr:
    nor_pre.append("Positive")
  elif "Neutral" in pr:
    nor_pre.append("Neutral")
  else:
    nor_pre.append("Unclassified")

pred_cot['Normalized Sentiment'] = nor_pre

In [ ]:
pred_cot['Normalized Sentiment'].value_counts()

,count
Normalized Sentiment,
Negative,334
Positive,109
Neutral,58


In [ ]:
pred_cot.to_excel('Qwen-SA-CoT.xlsx', index = False)

In [ ]:
y_true = data['sentiment'].values
print(classification_report(y_true, pred_cot['Normalized Sentiment'].values, digits = 4))

              precision    recall  f1-score   support

    Negative     0.7543    0.7904    0.7719       167
     Neutral     0.6748    0.6587    0.6667       167
    Positive     0.7669    0.7485    0.7576       167

    accuracy                         0.7325       501
   macro avg     0.7320    0.7325    0.7321       501
weighted avg     0.7320    0.7325    0.7321       501

